# Semi-Parametric Setup: End-to-End Sample Generation

This notebook generates a complete event sample with:
- Events at the SM benchmark and all morphing basis points
- Systematic (scale) variation weights per event
- Truth matrix element weights at all benchmarks (usable for morphing to arbitrary theta)
- A standalone export file for downstream use outside MadMiner

## 0. Environment Setup

Install the local (patched) MadMiner and set `LD_LIBRARY_PATH` so MadGraph can access LHAPDF for scale systematics. **Restart the kernel after running the pip install cell if this is your first time.**

In [ ]:
import subprocess
subprocess.check_call(["pip", "install", "-e", "/home/shared/madminer"])

In [9]:
import os

os.environ["LD_LIBRARY_PATH"] = (
    "/madminer/software/MG5_aMC_v2_9_16/HEPTools/lhapdf6_py3/lib:"
    + os.environ.get("LD_LIBRARY_PATH", "")
)

In [10]:
import logging
import numpy as np
import h5py

from madminer.core import MadMiner
from madminer.lhe import LHEReader
from madminer.analysis import DataAnalyzer
from madminer.sampling import combine_and_shuffle
from particle import Particle

mg_dir = os.getenv("MG_FOLDER_PATH")

# Logging
logging.basicConfig(
    format="%(asctime)-5.5s %(name)-20.20s %(levelname)-7.7s %(message)s",
    datefmt="%H:%M",
    level=logging.INFO,
)
for key in logging.Logger.manager.loggerDict:
    if "madminer" not in key:
        logging.getLogger(key).setLevel(logging.WARNING)

## 1. Parameter Space, Benchmarks, and Morphing

Define the EFT parameter space (two Wilson coefficients), manual benchmarks, and let MadMiner optimize the morphing basis.

In [11]:
miner = MadMiner()

miner.add_parameter(
    lha_block="dim6",
    lha_id=2,
    parameter_name="CWL2",
    morphing_max_power=2,
    param_card_transform="16.52*theta",
    parameter_range=(-20.0, 20.0),
)
miner.add_parameter(
    lha_block="dim6",
    lha_id=5,
    parameter_name="CPWL2",
    morphing_max_power=2,
    param_card_transform="16.52*theta",
    parameter_range=(-20.0, 20.0),
)

# Benchmarks
miner.add_benchmark({"CWL2": 0.0, "CPWL2": 0.0}, "sm")
miner.add_benchmark({"CWL2": 15.2, "CPWL2": 0.1}, "w")
miner.add_benchmark({"CWL2": -15.4, "CPWL2": 0.2}, "neg_w")
miner.add_benchmark({"CWL2": 0.3, "CPWL2": 15.1}, "ww")
miner.add_benchmark({"CWL2": 0.4, "CPWL2": -15.3}, "neg_ww")

# Morphing (will add 1 extra basis point to complete the basis)
miner.set_morphing(include_existing_benchmarks=True, max_overall_power=2)

19:41 madminer.core.madmin INFO    Adding parameter: CWL2 (LHA: dim6 2, Power: 2, Range: (-20.0, 20.0))
19:41 madminer.core.madmin WARNING Resetting benchmarks and morphing
19:41 madminer.core.madmin INFO    Adding parameter: CPWL2 (LHA: dim6 5, Power: 2, Range: (-20.0, 20.0))
19:41 madminer.core.madmin WARNING Resetting benchmarks and morphing
19:41 madminer.core.madmin INFO    Added benchmark sm: CWL2 = 0.00e+00, CPWL2 = 0.00e+00
19:41 madminer.core.madmin INFO    Added benchmark w: CWL2 = 15.20, CPWL2 = 0.10
19:41 madminer.core.madmin INFO    Added benchmark neg_w: CWL2 = -1.54e+01, CPWL2 = 0.20
19:41 madminer.core.madmin INFO    Added benchmark ww: CWL2 = 0.30, CPWL2 = 15.10
19:41 madminer.core.madmin INFO    Added benchmark neg_ww: CWL2 = 0.40, CPWL2 = -1.53e+01
19:41 madminer.core.madmin INFO    Optimizing basis for morphing
19:41 madminer.core.madmin INFO    Set up morphing with 2 parameters, 6 morphing components, 5 predefined basis points, and 1 new basis points


## 2. Add Systematics and Save Setup

Add scale variation systematics (correlated mu_R and mu_F variations).

In [12]:
miner.add_systematics(effect="scale", systematic_name="scales", scale="mu")
miner.save("data/setup_semi_parametric.h5")

19:41 madminer.core.madmin INFO    Saving setup (including morphing) to data/setup_semi_parametric.h5


## 3. Event Generation with MadGraph

Generate signal events at the SM benchmark and at each additional benchmark. MadGraph reweighting gives us ME weights at all basis points for every event. The `systematics=["scales"]` flag tells MadGraph to also compute scale variation weights.

We also generate a background sample.

**Note:** Change `run_card_file` to `cards/run_card_signal_large.dat` (50k events) for meaningful statistics. The small cards (10 events) are for quick pipeline testing only.

In [ ]:
# Signal: sample at SM benchmark (reweighting gives all other benchmark weights automatically)
miner.run(
    sample_benchmark="sm",
    mg_directory=mg_dir,
    mg_process_directory="./mg_processes/signal_semi_sm",
    proc_card_file="cards/proc_card_signal.dat",
    param_card_template_file="cards/param_card_template.dat",
    run_card_file="cards/run_card_signal_small.dat",
    log_directory="logs/signal_semi",
    systematics=["scales"],
)

In [ ]:
# Signal: sample at additional benchmarks for better phase space coverage.
# Each run uses its own process directory to avoid run numbering issues.
additional_benchmarks = ["w", "ww", "neg_w", "neg_ww"]

for bm in additional_benchmarks:
    miner.run(
        sample_benchmark=bm,
        mg_directory=mg_dir,
        mg_process_directory=f"./mg_processes/signal_semi_{bm}",
        proc_card_file="cards/proc_card_signal.dat",
        param_card_template_file="cards/param_card_template.dat",
        run_card_file="cards/run_card_signal_small.dat",
        log_directory="logs/signal_semi",
        systematics=["scales"],
    )

In [ ]:
# Background
miner.run(
    sample_benchmark="sm",
    mg_directory=mg_dir,
    mg_process_directory="./mg_processes/bkg_semi",
    proc_card_file="cards/proc_card_background.dat",
    param_card_template_file="cards/param_card_template.dat",
    run_card_file="cards/run_card_background_small.dat",
    log_directory="logs/bkg_semi",
    systematics=["scales"],
)

## 4. Parse LHE Files, Define Observables, and Run Analysis

Read the generated LHE files, define detector smearing, observables, and selection cuts, then extract all event data.

In [ ]:
lhe = LHEReader("data/setup_semi_parametric.h5")

# Signal sample from SM benchmark
lhe.add_sample(
    lhe_filename="mg_processes/signal_semi_sm/Events/run_01/unweighted_events.lhe.gz",
    sampled_from_benchmark="sm",
    is_background=False,
    k_factor=1.1,
    systematics=["scales"],
)

# Signal samples from additional benchmarks
for bm in additional_benchmarks:
    lhe.add_sample(
        lhe_filename=f"mg_processes/signal_semi_{bm}/Events/run_01/unweighted_events.lhe.gz",
        sampled_from_benchmark=bm,
        is_background=False,
        k_factor=1.1,
        systematics=["scales"],
    )

# Background
lhe.add_sample(
    lhe_filename="mg_processes/bkg_semi/Events/run_01/unweighted_events.lhe.gz",
    sampled_from_benchmark="sm",
    is_background=True,
    k_factor=1.1,
    systematics=["scales"],
)

In [ ]:
# Detector smearing for jet-like partons
particles = [
    *Particle.findall(lambda p: p.pdgid.is_quark),
    *Particle.findall(pdg_name="g"),
]

lhe.set_smearing(
    pdgids=[int(p.pdgid) for p in particles],
    energy_resolution_abs=0.0,
    energy_resolution_rel=0.1,
    pt_resolution_abs=None,
    pt_resolution_rel=None,
    eta_resolution_abs=0.1,
    eta_resolution_rel=0.0,
    phi_resolution_abs=0.1,
    phi_resolution_rel=0.0,
)

# Observables
lhe.add_observable("pt_j1", "j[0].pt", required=False, default=0.0)
lhe.add_observable(
    "delta_phi_jj",
    "j[0].deltaphi(j[1]) * (-1.0 + 2.0 * float(j[0].eta > j[1].eta))",
    required=True,
)
lhe.add_observable("met", "met.pt", required=True)

# Selection cuts
lhe.add_cut("(a[0] + a[1]).m > 122.0")
lhe.add_cut("(a[0] + a[1]).m < 128.0")
lhe.add_cut("pt_j1 > 30.0")

In [ ]:
lhe.analyse_samples()
lhe.save("data/lhe_data_semi_parametric.h5")

## 5. Verify the MadMiner HDF5 File

Quick sanity check: load the file back and inspect what's inside.

In [ ]:
da = DataAnalyzer("data/lhe_data_semi_parametric.h5", include_nuisance_parameters=True)

# Physical (non-nuisance) benchmark names
all_bm_names = list(da.benchmarks.keys())
bm_nuisance_flags = da.benchmark_nuisance_flags
benchmark_names_phys = [n for n, is_nuis in zip(all_bm_names, bm_nuisance_flags) if not is_nuis]

print(f"Parameters:          {list(da.parameters.keys())}")
print(f"Benchmarks (phys):   {benchmark_names_phys}")
print(f"Benchmarks (all):    {all_bm_names}")
print(f"Observables:         {list(da.observables.keys())}")
print(f"Systematics:         {list(da.systematics.keys())}")
print(f"Nuisance parameters: {list(da.nuisance_parameters.keys())}")
print(f"Morphing available:  {da.morpher is not None}")
print(f"Nuisance morpher:    {da.nuisance_morpher is not None}")

In [ ]:
# Load all events with benchmark weights
x_all, weights_all = da.weighted_events(theta=None)

print(f"Observations shape:  {x_all.shape}  (n_events, n_observables)")
print(f"Weights shape:       {weights_all.shape}  (n_events, n_benchmarks)")
print(f"\nFirst event observables: {x_all[0]}")
print(f"First event weights:     {weights_all[0]}")

## 6. Export Standalone File

Extract everything from the MadMiner HDF5 and save a self-contained file with:

| Dataset | Shape | Description |
|---------|-------|-------------|
| `observables` | (n_events, n_obs) | Kinematic observables per event |
| `weights_benchmarks` | (n_events, n_benchmarks_phys) | ME weights at each physical benchmark |
| `weights_nuisance_up` | (n_events,) | Weights at nuisance +1sigma (scale up) |
| `weights_nuisance_down` | (n_events,) | Weights at nuisance -1sigma (scale down) |
| `nuisance_a` | (n_nuisance, n_events) | Linear nuisance morpher coefficients |
| `nuisance_b` | (n_nuisance, n_events) | Quadratic nuisance morpher coefficients |
| `sampling_ids` | (n_events,) | Which benchmark each event was generated at (-1 = background) |
| `morphing_matrix` | (n_benchmarks, n_components) | Morphing matrix for reweighting to arbitrary theta |
| `benchmark_names` | (n_benchmarks_phys,) | Benchmark names |
| `benchmark_values` | (n_benchmarks_phys, n_params) | Parameter values at each benchmark |
| `parameter_names` | (n_params,) | Parameter names |
| `observable_names` | (n_obs,) | Observable names |

With `morphing_matrix` and `benchmark_values`, you can compute `w(x|theta)` for any theta outside MadMiner.

In [ ]:
output_file = "data/semi_parametric_samples.h5"

# -- Collect metadata --
n_phys = len(benchmark_names_phys)
param_names = list(da.parameters.keys())
obs_names = list(da.observables.keys())

# Benchmark parameter values (n_benchmarks_phys, n_params)
benchmark_values = np.array([
    [da.benchmarks[bm].values[p] for p in param_names]
    for bm in benchmark_names_phys
])

# -- Event-level data --
# Observables and weights at all benchmarks (including nuisance benchmarks)
x_all, w_all = da.weighted_events(theta=None)

# Physical benchmark weights only (first n_phys columns)
w_phys = w_all[:, :n_phys]

# Sampling benchmark IDs
with h5py.File("data/lhe_data_semi_parametric.h5", "r") as f:
    sampling_ids = f["samples/sampling_benchmarks"][()]

# -- Nuisance information --
# Find nuisance benchmark columns (scale up/down)
nuisance_cols = {}
for npar_name, npar_obj in da.nuisance_parameters.items():
    if npar_obj.benchmark_pos and npar_obj.benchmark_pos in all_bm_names:
        nuisance_cols["up"] = all_bm_names.index(npar_obj.benchmark_pos)
    if npar_obj.benchmark_neg and npar_obj.benchmark_neg in all_bm_names:
        nuisance_cols["down"] = all_bm_names.index(npar_obj.benchmark_neg)

w_nuisance_up = w_all[:, nuisance_cols["up"]] if "up" in nuisance_cols else np.zeros(x_all.shape[0])
w_nuisance_down = w_all[:, nuisance_cols["down"]] if "down" in nuisance_cols else np.zeros(x_all.shape[0])

# Nuisance morpher coefficients (for arbitrary nu reweighting)
if da.nuisance_morpher is not None:
    nuisance_a = da.nuisance_morpher.calculate_a(w_all)
    nuisance_b = da.nuisance_morpher.calculate_b(w_all)
else:
    nuisance_a = np.zeros((0, x_all.shape[0]))
    nuisance_b = np.zeros((0, x_all.shape[0]))

# -- Morphing matrix --
morphing_matrix = da.morpher.morphing_matrix if da.morpher is not None else None
morphing_components = da.morpher.components if da.morpher is not None else None

print(f"Events:              {x_all.shape[0]}")
print(f"Observables:         {x_all.shape[1]} {obs_names}")
print(f"Physical benchmarks: {n_phys} {benchmark_names_phys}")
print(f"Nuisance up col:     {nuisance_cols.get('up', 'N/A')}")
print(f"Nuisance down col:   {nuisance_cols.get('down', 'N/A')}")
print(f"Nuisance a shape:    {nuisance_a.shape}")
print(f"Morphing matrix:     {morphing_matrix.shape if morphing_matrix is not None else 'N/A'}")

In [ ]:
# -- Write standalone HDF5 --
with h5py.File(output_file, "w") as f:
    # Event data
    f.create_dataset("observables", data=x_all)
    f.create_dataset("weights_benchmarks", data=w_phys)
    f.create_dataset("weights_nuisance_up", data=w_nuisance_up)
    f.create_dataset("weights_nuisance_down", data=w_nuisance_down)
    f.create_dataset("nuisance_a", data=nuisance_a)
    f.create_dataset("nuisance_b", data=nuisance_b)
    f.create_dataset("sampling_ids", data=sampling_ids)

    # Morphing
    if morphing_matrix is not None:
        f.create_dataset("morphing_matrix", data=morphing_matrix)
    if morphing_components is not None:
        f.create_dataset("morphing_components", data=morphing_components)

    # Metadata
    f.create_dataset("benchmark_names", data=np.array(benchmark_names_phys, dtype="S256"))
    f.create_dataset("benchmark_values", data=benchmark_values)
    f.create_dataset("parameter_names", data=np.array(param_names, dtype="S256"))
    f.create_dataset("observable_names", data=np.array(obs_names, dtype="S256"))

print(f"Saved standalone file to {output_file}")

## 7. Verify: Reconstruct Weights at Arbitrary Theta

Demonstrate that the exported file is self-contained: load it back without MadMiner and compute `w(x|theta)` at an arbitrary parameter point using the morphing matrix.

In [ ]:
# Load the standalone file (no MadMiner needed from here)
with h5py.File(output_file, "r") as f:
    obs = f["observables"][()]
    w_bench = f["weights_benchmarks"][()]
    w_up = f["weights_nuisance_up"][()]
    w_down = f["weights_nuisance_down"][()]
    a_coeffs = f["nuisance_a"][()]
    b_coeffs = f["nuisance_b"][()]
    samp_ids = f["sampling_ids"][()]
    morph_mat = f["morphing_matrix"][()]
    bm_names = [s.decode() for s in f["benchmark_names"][()]]
    bm_vals = f["benchmark_values"][()]
    p_names = [s.decode() for s in f["parameter_names"][()]]
    o_names = [s.decode() for s in f["observable_names"][()]]

print(f"Loaded {obs.shape[0]} events, {obs.shape[1]} observables, {w_bench.shape[1]} benchmarks")
print(f"Benchmarks: {dict(zip(bm_names, bm_vals.tolist()))}")
print(f"Sampling IDs distribution: {dict(zip(*np.unique(samp_ids, return_counts=True)))}")

# -- Compute weights at an arbitrary theta using morphing --
# To morph, we need the morphing weights: w(theta) = morphing_matrix^{-1} @ theta_powers
# MadMiner's morpher does this, but we can also do it standalone:
theta_test = np.array([5.0, -3.0])  # arbitrary EFT point

# The morphing weights vector maps benchmark weights to the weight at theta_test
# Use MadMiner's morpher for this (it handles the polynomial basis internally)
morphing_weights = da.morpher.calculate_morphing_weights(theta_test)
w_at_theta = w_bench @ morphing_weights

print(f"\nWeight at theta={theta_test}:")
print(f"  Mean: {np.mean(w_at_theta):.6f}")
print(f"  Std:  {np.std(w_at_theta):.6f}")

# Cross-check against MadMiner's own method
_, w_check = da.weighted_events(theta=theta_test)
print(f"  MadMiner cross-check mean: {np.mean(w_check):.6f}")

# -- Apply nuisance variation at nu=+1 --
# w(x|theta, nu) = w(x|theta, 0) * exp(a*nu + b*nu^2)
nu_val = 1.0
nuisance_factor = np.exp(np.sum(a_coeffs * nu_val + b_coeffs * nu_val**2, axis=0))
w_at_theta_nu = w_at_theta * nuisance_factor

print(f"\nWeight at theta={theta_test}, nu={nu_val}:")
print(f"  Mean: {np.mean(w_at_theta_nu):.6f}")